# 04 — Building Morphology

**Purpose:** download building footprints for the Oxford Street study area, derive
building heights from LIDAR, validate the result, and compute morphological
descriptors (area, compactness, elongation, orientation, shared walls, volume,
storey estimate) for buildings in the immediate frontage of the street/artery
network.

**Data sources**
- Footprints: OpenStreetMap, via `osmnx` (`POI_BBOX` extent)
- Heights: Environment Agency National LIDAR Programme, 1m composite tiles,
  first-return DSM + DTM, height = DSM − DTM per building footprint
- *(Digimap / OS MasterMap Building Heights was the preferred source — richer,
  surveyed heights rather than a LIDAR-surface approximation — but institutional
  access was still pending at the time this notebook was built. If access comes
  through, treat MasterMap as the primary height source and this LIDAR pipeline
  as the interim/validation dataset; footprint IDs won't match 1:1 across
  sources, so switching means re-running frontage assignment, not just the
  height column.)*

**Status**
- Footprint download, height derivation, and QA: **working**
- Frontage assignment (tessellation) and morphological metrics: **not yet run
  in this notebook** — depends on a `momepy` tessellation step against
  `streets_gdf` (core + arteries) that lives in an earlier stage of the
  pipeline. The metrics cell below is written and ready but will raise
  `NameError` until `frontage_buildings` exists upstream.

**Known environment issues** (see `HANDOFF-03.md` for the running list)
- `rasterio.merge()` silently returns all-zero rasters on this environment
  (rasterio 1.4.4) when mosaicking the DSM/DTM tiles — confirmed via isolated
  single-tile tests. Worked around below by running `zonal_stats` per source
  tile and reassembling results in pandas, rather than mosaicking first.
- Basemap tiles via `contextily` currently render an "API key required"
  watermark regardless of provider (CartoDB tried via dict, string URL, and
  `xyzservices` object; cache cleared; direct `requests.get` to the tile CDN
  returns a 200 but was never visually confirmed as a real tile). Root cause
  unresolved — deprioritised as non-blocking. Maps below will still show the
  watermark until this is chased down.
- `floors_est` (below) uses a fixed 3.5m-per-floor assumption — a hand-tuned
  constant in the same category as the angular-tracing thresholds in
  `config.py`, and needs the same sensitivity-analysis treatment before
  publication.


## Setup

In [ ]:
import os
import glob

import numpy as np
import pandas as pd
import geopandas as gpd
import osmnx as ox
import rasterio
from rasterstats import zonal_stats
from shapely.geometry import box
import matplotlib.pyplot as plt
import contextily as cx
import momepy

from config import POI_BBOX, PROJECTED_CRS, LIDAR_DIR, DTM_DIR, DSM_DIR, OUTPUTS_DIR


## 1. Building footprints (OSM)

Pulls all `building=*` polygon features within `POI_BBOX` and reprojects to
`PROJECTED_CRS` (EPSG:32630) for consistency with the rest of the pipeline's
geometry work (area, length, corridor/artery tracing tolerances).


In [ ]:
buildings = ox.features_from_bbox(bbox=POI_BBOX, tags={"building": True})
buildings = buildings[buildings.geometry.type.isin(["Polygon", "MultiPolygon"])].copy()
buildings = buildings.to_crs(PROJECTED_CRS)

print(f"buildings downloaded: {len(buildings)}")


## 2. Building height from LIDAR

**DTM** (Digital Terrain Model) = bare-earth ground elevation.
**DSM** (Digital Surface Model) = elevation of the first surface a LIDAR pulse
hits — roofline, canopy, etc. First-return DSM specifically, since first
return is the top-of-roof bounce (last-return is for canopy-penetration work,
not relevant here).

```
height_m = DSM − DTM
```

averaged per building footprint. This is a surface approximation, not a
surveyed height — it can run slightly high on complex rooflines (parapets,
plant housing) and is noisy at tile seams. Flagged as an interim proxy pending
MasterMap Building Heights access (see intro).

Three non-adjacent OS grid tiles cover the study area
(`TQ27ne` / `TQ28se` / `TQ38sw`). Because `rasterio.merge()` is broken in this
environment for these files (see intro), height is computed per tile and
reassembled, rather than mosaicking first. This also means no building can
span more than one tile — a side benefit of the tiles being non-adjacent.


In [ ]:
dsm_tiles = glob.glob(os.path.join(DSM_DIR, "*.tif"))
dtm_tiles = glob.glob(os.path.join(DTM_DIR, "*.tif"))

print(f"DSM tiles found: {len(dsm_tiles)}")
print(f"DTM tiles found: {len(dtm_tiles)}")
assert len(dsm_tiles) == 3, f"expected 3 DSM tiles, found {len(dsm_tiles)}"
assert len(dtm_tiles) == 3, f"expected 3 DTM tiles, found {len(dtm_tiles)}"


In [ ]:
def zonal_stats_by_tile(gdf, tile_paths, stat="mean"):
    """Run zonal_stats per raster tile and reassemble into one Series.

    Avoids rasterio.merge(), which returns all-zero mosaics for these tiles
    on this environment (rasterio 1.4.4) — see notebook intro.
    """
    result = pd.Series(index=gdf.index, dtype="float64")
    for tile_path in tile_paths:
        with rasterio.open(tile_path) as src:
            tile_box = box(*src.bounds)
            nodata = src.nodata
        subset = gdf[gdf.geometry.intersects(tile_box)]
        if len(subset) == 0:
            continue
        stats = zonal_stats(subset, tile_path, stats=stat, nodata=nodata)
        vals = np.array(
            [s[stat] if s[stat] is not None else np.nan for s in stats],
            dtype="float64",
        )
        result.loc[subset.index] = vals
    return result


# EA LIDAR tiles are in British National Grid (EPSG:27700), not PROJECTED_CRS
# (EPSG:32630) — reproject for the zonal stats step only, so geometries
# actually overlap the raster. height_m is assigned back onto `buildings`,
# which stays in PROJECTED_CRS throughout.
buildings_bng = buildings.to_crs("EPSG:27700")

dsm_vals = zonal_stats_by_tile(buildings_bng, dsm_tiles)
dtm_vals = zonal_stats_by_tile(buildings_bng, dtm_tiles)

buildings["height_m"] = dsm_vals - dtm_vals

print(f"missing: {buildings['height_m'].isna().sum()} / {len(buildings)}")
print(buildings["height_m"].describe())


## 3. QA — validate heights

Two checks:
1. **Missing-height buildings**, mapped — should cluster at the edges of the
   3 tile footprints (buildings outside all 3 tiles have no LIDAR coverage by
   construction), not scatter randomly through the middle of the study area.
2. **Height distribution + outliers over 60m**, mapped individually against
   the basemap. Central London has a handful of genuinely tall buildings
   (e.g. Centre Point, ~117m, at the east end of Oxford Street) — the point
   is to separate those from tile-seam or footprint artifacts, not to assume
   every outlier is a bug.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 10))

missing = buildings[buildings["height_m"].isna()]
present = buildings[buildings["height_m"].notna()]

present.plot(ax=axes[0], color="lightgrey", markersize=0.1, label="has height")
missing.plot(ax=axes[0], color="red", markersize=0.1, label="missing")
axes[0].set_title(f"Missing height coverage ({len(missing)} / {len(buildings)})")
axes[0].legend()

buildings["height_m"].hist(ax=axes[1], bins=100)
axes[1].axvline(50, color="orange", linestyle="--", label="50m")
axes[1].axvline(100, color="red", linestyle="--", label="100m")
axes[1].set_title("Height distribution")
axes[1].set_xlabel("Height (m)")
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, "height_diagnostics.png"), dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# individual outlier check, >60m
tall = buildings[buildings["height_m"] > 60].sort_values("height_m", ascending=False)
print(f"{len(tall)} buildings over 60m")

if len(tall) > 0:
    fig, ax = plt.subplots(figsize=(12, 12))
    buildings.plot(ax=ax, color="lightgrey", edgecolor="grey", linewidth=0.2)
    tall.plot(ax=ax, color="red", edgecolor="black")

    xmin, ymin, xmax, ymax = tall.total_bounds
    pad = 200  # metres of context around the outliers
    ax.set_xlim(xmin - pad, xmax + pad)
    ax.set_ylim(ymin - pad, ymax + pad)

    # NOTE: basemap currently renders an "API key required" watermark on this
    # environment regardless of provider — unresolved, see notebook intro.
    cx.add_basemap(ax, crs=buildings.crs, source=cx.providers.CartoDB.Positron)
    ax.set_xlim(xmin - pad, xmax + pad)  # re-apply — contextily resets limits on tile fetch
    ax.set_ylim(ymin - pad, ymax + pad)

    ax.set_title(f"{len(tall)} buildings > 60m — visual check")
    plt.savefig(os.path.join(OUTPUTS_DIR, "tall_outliers_check.png"), dpi=150, bbox_inches="tight")
    plt.show()


## 4. Visualize — building height choropleth

Continuous colour ramp, clipped to the 2nd/98th percentile so a handful of
tall outliers (Centre Point etc.) don't compress the rest of the study area
into one indistinguishable shade at the low end of the scale.


In [ ]:
plot_data = buildings[buildings["height_m"].notna()].copy()

fig, ax = plt.subplots(figsize=(16, 16))

plot_data.plot(
    ax=ax,
    column="height_m",
    cmap="viridis",
    legend=True,
    legend_kwds={"label": "Building height (m)", "shrink": 0.5},
    edgecolor="none",
    vmin=plot_data["height_m"].quantile(0.02),
    vmax=plot_data["height_m"].quantile(0.98),
)

xmin, ymin, xmax, ymax = plot_data.total_bounds

# NOTE: basemap currently renders an "API key required" watermark on this
# environment regardless of provider — unresolved, see notebook intro.
cx.add_basemap(
    ax,
    crs=plot_data.crs,
    source="https://basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png",
)
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)  # re-apply — contextily resets limits on tile fetch

ax.set_axis_off()
ax.set_title("Building heights — Oxford Street study area", fontsize=15)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, "building_heights_gradual.png"), dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
buildings.to_file(os.path.join(OUTPUTS_DIR, "buildings_with_height.gpkg"), driver="GPKG")

## 5. Morphological metrics (frontage buildings)

**Pending:** this cell depends on `frontage_buildings` — the subset of
`buildings` assigned to street/artery frontage via `momepy` tessellation
(cell footprint → tessellation → nearest-street assignment → filter to cells
touching a street/artery edge). That step hasn't been run in this notebook
yet; it needs `streets_gdf` (core Oxford Street + arteries, with a
`segment_type` column distinguishing them) from the corridor-tracing stage.

Once `frontage_buildings` exists upstream, this cell computes the standard
`momepy` descriptor set plus a derived storey estimate. `floors_est` uses a
fixed 3.5m/floor assumption — flag alongside the angular-threshold caveats
for the methodology writeup if it feeds anything downstream.


In [ ]:
assert "frontage_buildings" in dir(), (
    "frontage_buildings not defined — run the tessellation-based frontage "
    "assignment step first (momepy.Tessellation + get_nearest_street against "
    "streets_gdf), see markdown above."
)

frontage_buildings["area"] = frontage_buildings.geometry.area
frontage_buildings["perimeter"] = frontage_buildings.geometry.length
frontage_buildings["compactness"] = momepy.CircularCompactness(frontage_buildings).series
frontage_buildings["elongation"] = momepy.Elongation(frontage_buildings).series
frontage_buildings["orientation"] = momepy.Orientation(frontage_buildings).series
frontage_buildings["shared_walls"] = momepy.SharedWalls(frontage_buildings).series
frontage_buildings["volume"] = frontage_buildings["area"] * frontage_buildings["height_m"]
frontage_buildings["floors_est"] = (frontage_buildings["height_m"] / 3.5).round()  # coarse; flag as assumption
